# ⚡ Electricity Consumption Analysis & Theft Detection
## Phase 1: Data Understanding & Exploratory Time-Series Analysis

### Project Overview
Electricity theft (Non-Technical Loss - NTL) accounts for billions of dollars in lost utility revenue worldwide. Common theft mechanisms include meter tampering, phase bypassing, shunt connections, and meter clock manipulation.

In this notebook, we analyze the State Grid Corporation of China (SGCC) time-series dataset. Each consumer record contains **1,034 continuous daily electricity consumption readings** spanning **January 1, 2014 to October 31, 2016**.

### Key Analysis Goals:
1. **Dataset Structure & Time-Series Dimensionality**: Inspect continuous temporal readings.
2. **Class Imbalance**: Quantify the Normal (0) vs. Fraud/Theft (1) ratio.
3. **Missing Values & Zero-Consumption Anomalies**: Detect meter dropouts and prolonged zero periods.
4. **Time-Series Profiling**: Compare aggregate trajectories and individual consumer electricity profiles.
5. **Theft Signatures**: Identify domain signatures such as sudden drops, suppressed baselines, and erratic variance.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully.")


### 1. Dataset Loading & Dimensionality Inspection

In [ ]:
# Load sample consumer dataset
df = pd.read_csv('../data/sample_test_consumers.csv')

print(f"Dataset Shape: {df.shape[0]} consumers x {df.shape[1]} columns")
non_date_cols = [c for c in df.columns if '/' not in c]
date_cols = [c for c in df.columns if '/' in c]
print(f"Metadata columns: {non_date_cols}")
print(f"Total time-series reading days: {len(date_cols)} (From {date_cols[0]} to {date_cols[-1]})")
df[['CONS_NO', 'FLAG'] + date_cols[:5]].head()


### 2. Class Distribution (Target Analysis)
The SGCC dataset has a natural class imbalance where theft instances account for ~8.5% in the full population.


In [ ]:
class_counts = df['FLAG'].value_counts()
class_pct = df['FLAG'].value_counts(normalize=True) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#10b981', '#ef4444']

# Bar chart
sns.barplot(x=['Normal (0)', 'Theft (1)'], y=class_counts.values, palette=colors, ax=ax1)
ax1.set_title('Consumer Count by Category', fontsize=13, fontweight='bold')
ax1.set_ylabel('Number of Consumers')
for i, v in enumerate(class_counts.values):
    ax1.text(i, v + 0.3, str(v), ha='center', fontweight='bold')

# Pie chart
ax2.pie(class_counts, labels=['Normal Consumers', 'Theft Cases'], autopct='%1.1f%%', 
        colors=colors, explode=(0, 0.08), startangle=140, textprops={'fontweight': 'bold'})
ax2.set_title('Class Ratio in Evaluation Sample', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()


### 3. Missing Readings & Zero-Consumption Days
A primary signature of meter tampering is an abnormal run of zero readings while the premises remain occupied.


In [ ]:
# Compute zero consumption metrics
zero_counts = (df[date_cols] == 0).sum(axis=1)
df['zero_pct'] = (zero_counts / len(date_cols)) * 100
df['mean_consumption'] = df[date_cols].mean(axis=1)
df['std_consumption'] = df[date_cols].std(axis=1)
df['max_consumption'] = df[date_cols].max(axis=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Boxplot of Zero Days %
sns.boxplot(x='FLAG', y='zero_pct', data=df, palette=['#10b981', '#ef4444'], ax=ax1)
ax1.set_xticklabels(['Normal Consumer', 'Theft Consumer'], fontweight='bold')
ax1.set_title('Percentage of Zero-Consumption Days', fontsize=13, fontweight='bold')
ax1.set_ylabel('Zero-Reading Days (%)')

# Boxplot of Mean Consumption
sns.boxplot(x='FLAG', y='mean_consumption', data=df, palette=['#10b981', '#ef4444'], ax=ax2)
ax2.set_xticklabels(['Normal Consumer', 'Theft Consumer'], fontweight='bold')
ax2.set_title('Average Daily Consumption (kWh)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Mean Daily kWh')

plt.tight_layout()
plt.show()

print("Summary Statistics by Category:")
print(df.groupby('FLAG')[['zero_pct', 'mean_consumption', 'std_consumption']].mean())


### 4. Aggregate Time-Series Trajectories
Let us compare the continuous 1,034-day average electricity consumption trajectory between Normal and Fraudulent consumers.


In [ ]:
normal_mean_ts = df[df['FLAG'] == 0][date_cols].mean(axis=0)
theft_mean_ts = df[df['FLAG'] == 1][date_cols].mean(axis=0)

plt.figure(figsize=(16, 6))
plt.plot(normal_mean_ts.values, label='Normal Consumers (Avg Daily kWh)', color='#10b981', linewidth=1.8, alpha=0.85)
plt.plot(theft_mean_ts.values, label='Theft Consumers (Avg Daily kWh)', color='#ef4444', linewidth=1.8, alpha=0.85)

# Calculate 30-day rolling averages
plt.plot(normal_mean_ts.rolling(30).mean().values, label='Normal 30-Day Trend', color='#047857', linestyle='--', linewidth=2.2)
plt.plot(theft_mean_ts.rolling(30).mean().values, label='Theft 30-Day Trend', color='#b91c1c', linestyle='--', linewidth=2.2)

plt.title('1,034-Day Aggregate Electricity Consumption Trajectory (2014 - 2016)', fontsize=14, fontweight='bold')
plt.xlabel('Day Index (0 to 1033)')
plt.ylabel('Daily Electricity Consumption (kWh)')
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()


### 5. Individual Consumer Profiles & Theft Signatures
We contrast 3 normal consumers exhibiting regular seasonal cycles against 3 fraudulent consumers exhibiting sudden sustained drops, prolonged flatlines, or abnormal bypass drops.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=True)

normal_samples = df[df['FLAG'] == 0].head(3)
theft_samples = df[df['FLAG'] == 1].head(3)

for idx, (_, row) in enumerate(normal_samples.iterrows()):
    ax = axes[idx, 0]
    ax.plot(row[date_cols].values, color='#10b981', lw=1.2)
    ax.set_title(f'Normal: {row["CONS_NO"][:14]}... (Consistent pattern)', fontsize=10, fontweight='bold')
    ax.set_ylabel('kWh')

for idx, (_, row) in enumerate(theft_samples.iterrows()):
    ax = axes[idx, 1]
    ax.plot(row[date_cols].values, color='#ef4444', lw=1.2)
    ax.set_title(f'Theft: {row["CONS_NO"][:14]}... (Sudden drop / flatline)', fontsize=10, fontweight='bold')
    ax.set_ylabel('kWh')

axes[2, 0].set_xlabel('Day Index')
axes[2, 1].set_xlabel('Day Index')

plt.suptitle('Comparison of Individual Normal vs. Theft Consumption Profiles', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


### 6. Conclusions & Domain Insights
1. **Seasonal Consistency vs. Disruption**: Normal consumers show distinct seasonal cycles (e.g., summer air-conditioning and winter heating peaks). Theft consumers show sudden disruption of seasonal cycles.
2. **Zero-Day Ratio**: Theft consumers demonstrate significantly higher zero-consumption days and abrupt sustained drops in recorded load.
3. **Feature Representations**: The full 1,034 temporal sequence provides ample dynamic signal for non-linear gradient boosted tree classifiers (XGBoost).
